In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_14.pickle

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

### cell 15 (optimized for cudf)

def bar_chart_multiple_choice_24(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    # response_counts is already a cudf.Series
    response_counts.index.name = ''
    # write top choices directly from the cudf.Series without converting back to pandas
    response_counts.iloc[:num_choices]
                    .to_frame()
                    .to_csv(
        '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results'
        '/kaggle/working/individual_charts/data/' + title + '.csv',
        index=True
    )
    # compute max for chart scaling
    tmp_cmp = (response_counts.iloc[:num_choices] * 1.2).max()

# question we care about
question_of_interest = 'Are you currently a student? (high school, university, or graduate)'
# cache the two columns once to avoid repeated DataFrame.__getitem__ calls
country_col = responses_df_2022['In which country do you currently reside?']
student_col = responses_df_2022[question_of_interest]

for country, chart_title in [
    ('United States of America', 'Students status for Kaggle Survey participants from the USA'),
    ('India',                         'Students status for Kaggle Survey participants from India')
]:
    # build a boolean mask on the GPU
    mask = (country_col == country)
    # compute percentages in one pass (normalize=True) and stay on GPU
    percentages = (
        student_col[mask]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(1)
        .sort_index()
    )
    bar_chart_multiple_choice_24(
        response_counts=percentages,
        title=chart_title,
        y_axis_title='% of respondents',
        orientation=orientation_for_chart
    )


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_15_try_9.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_15_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[15], f)


In [ ]:
opt_output = Out.get(4)